In [56]:
import mlflow
import pandas as pd
import numpy as np
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import dagshub
import logging
import time
import os 

In [38]:
df = pd.read_csv('IMDB.csv')

In [39]:
df = df.sample(500)

In [40]:
df.to_csv('Movie Reviews.csv',index=False)

In [41]:
df = pd.read_csv("Movie Reviews.csv")

In [42]:
df.shape

(500, 2)

In [43]:
df.head()

,review,sentiment
0,With this movie I was really hoping that the i...,negative
1,I want to state first that I am a Christian (a...,negative
2,"LIGHTS OF NEW YORK was the first ""all-taking"" ...",positive
3,Excellent documentary that still manages to sh...,positive
4,"Most people, when they think of expressionist ...",positive


# Data Preprocessing

In [44]:
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [45]:
df = normalize_text(df)

In [46]:
df.head()

,review,sentiment
0,movie really hoping idea make hashed together ...,negative
1,want state first christian and work film tv in...,negative
2,light new york first all taking feature film c...,positive
3,excellent documentary still manages shock enli...,positive
4,people think expressionist cinema look b w ger...,positive


In [47]:
df['sentiment'].value_counts()

sentiment
negative    268
positive    232
Name: count, dtype: int64

In [48]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [49]:
df.head()

,review,sentiment
0,movie really hoping idea make hashed together ...,negative
1,want state first christian and work film tv in...,negative
2,light new york first all taking feature film c...,positive
3,excellent documentary still manages shock enli...,positive
4,people think expressionist cinema look b w ger...,positive


In [50]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [51]:
df.head()

,review,sentiment
0,movie really hoping idea make hashed together ...,0
1,want state first christian and work film tv in...,0
2,light new york first all taking feature film c...,1
3,excellent documentary still manages shock enli...,1
4,people think expressionist cinema look b w ger...,1


In [52]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [53]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=0)

In [55]:
mlflow.set_tracking_uri('https://dagshub.com/ashokreddy304/Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning.mlflow')

dagshub.init(repo_owner = 'ashokreddy304',
             repo_name = 'Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning', 
             mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

Accessing as ashokreddy304

Initialized MLflow to track repo "ashokreddy304/Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning"

Repository ashokreddy304/Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning initialized!

2026/04/29 18:58:21 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3f65bdd4509842ea8f77ad74bdc786ed', creation_time=1777469302410, experiment_id='0', last_update_time=1777469302410, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

# Configure logging

In [57]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

2026-04-29 19:01:49,750 - INFO - Starting MLflow run...


In [60]:
with mlflow.start_run(run_name='logistic_regression_Bag of Words'):
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param('num_features',100)
        mlflow.log_param('test_size',0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)

        logging.info('Fitting the model...')
        model.fit(X_train,y_train)
        logging.info('Model training complete.')

        logging.info("Logging model parameters...")
        mlflow.log_param('model','LogisticRegression')

        logging.info('Making Predictions...')
        y_pred = model.predict(X_test)

        logging.info('Calculating evaluation metrics...')
        accuracy = accuracy_score(y_test,y_pred)
        precision = precision_score(y_test,y_pred)
        recall = recall_score(y_test,y_pred)
        f1 = f1_score(y_test,y_pred)

        logging.info('Logging evaluation metrics....')
        mlflow.log_metric('accuracy_score',accuracy)
        mlflow.log_metric('precision_score',precision)
        mlflow.log_metric('recall_score',recall)
        mlflow.log_metric('f1_score',f1)

        logging.info('Saving and Logging the model....')
        mlflow.sklearn.log_model(model,name='LogisticRegression')

        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

               # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-04-29 19:25:28,011 - INFO - Logging preprocessing parameters...
2026-04-29 19:25:29,287 - INFO - Initializing Logistic Regression model...
2026-04-29 19:25:29,292 - INFO - Fitting the model...
2026-04-29 19:25:29,379 - INFO - Model training complete.
2026-04-29 19:25:29,381 - INFO - Logging model parameters...
2026-04-29 19:25:29,925 - INFO - Making Predictions...
2026-04-29 19:25:29,925 - INFO - Calculating evaluation metrics...
2026-04-29 19:25:29,956 - INFO - Logging evaluation metrics....
2026-04-29 19:25:31,532 - INFO - Saving and Logging the model....
2026/04/29 19:25:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-04-29 19:25:51,761 - INFO

🏃 View run logistic_regression_Bag of Words at: https://dagshub.com/ashokreddy304/Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning.mlflow/#/experiments/0/runs/420ac061149e4ab594b92b805a8b194e
🧪 View experiment at: https://dagshub.com/ashokreddy304/Movie-Review-Sentiment-Analysis-using-NLP-Machine-Learning.mlflow/#/experiments/0
